In [2]:
# --- FULL SCRIPT (Re-Corrected CalculationSetup Instantiation) ---
import olca_ipc as olca
import olca_schema
import pandas as pd
import uuid
import traceback
import os

# Configuration
OLCA_PORT = 8080
LCIA_METHOD_NAME = "IPCC 2021 (ISO 14067)"
TARGET_IMPACT_CATEGORY = "IPCC 2021 (ISO 14067) - Total"
PROCESS_NAMES_FILEPATH = "all_process_names.txt" # Path to your text file

# extract_result_safely function remains the same
def extract_result_safely(client, calc_result_obj, target_category_name_str):
    carbon_footprint = None
    status = "Success"
    data_source_description = "Unknown source"
    if not calc_result_obj:
        print("              ❌ Calculation result object is None or empty.")
        return None, "Calculation Result Invalid/Empty"
    if hasattr(calc_result_obj, 'error') and calc_result_obj.error:
        error_msg = getattr(calc_result_obj.error, 'error', str(calc_result_obj.error))
        print(f"              ❌ Calculation failed with server error: {error_msg[:150]}")
        return None, f"Calc Server Error: {error_msg[:100]}"
    if not hasattr(calc_result_obj, 'uid') or not calc_result_obj.uid:
        if status == "Success":
            print("              ❌ Calculation result object has no UID, might be error indicator.")
            return None, "Calculation Result Invalid UID"
        return None, status
    if hasattr(calc_result_obj, 'wait_until_ready'):
        try:
            print("              ⏳ Waiting for calculation result to be ready...")
            calc_result_obj.wait_until_ready()
            if hasattr(calc_result_obj, 'error') and calc_result_obj.error:
                error_msg_after_wait = getattr(calc_result_obj.error, 'error', str(calc_result_obj.error))
                print(f"              ⚠️ Calculation result (after wait) has error: {error_msg_after_wait[:150]}")
                return None, f"Calc Error After Wait: {error_msg_after_wait[:100]}"
        except Exception as e_wait:
            print(f"              ⚠️ Error or issue during wait_until_ready: {e_wait}")
            print("              📊 Attempting to read results anyway...")
    fetched_impact_data = None
    if hasattr(calc_result_obj, 'get_total_impacts'):
        try:
            print("              🔍 Attempting to use 'get_total_impacts()'.")
            fetched_impact_data = calc_result_obj.get_total_impacts()
            data_source_description = "'get_total_impacts()'"
            if fetched_impact_data is not None:
                print(f"              📊 '{data_source_description}' call successful, returned {'data' if len(fetched_impact_data) > 0 else 'an empty list'}.")
            else:
                print(f"              📊 '{data_source_description}' returned None.")
        except Exception as e:
            print(f"              ⚠️ Error calling 'get_total_impacts()': {e}")
            fetched_impact_data = None
            status = f"Error calling {data_source_description}"
    else:
        print("              ℹ️  'get_total_impacts()' method not available.")
        if status == "Success": status = "get_total_impacts() not available"

    if not fetched_impact_data and status == "Success":
        print("              ℹ️  Trying alternative methods as primary failed/yielded no data.")
        if hasattr(calc_result_obj, 'impact_results') and calc_result_obj.impact_results is not None:
            fetched_impact_data = calc_result_obj.impact_results
            data_source_description = "'impact_results' attribute"
            print(f"              📊 Using '{data_source_description}'. Found {'data' if len(fetched_impact_data) > 0 else 'an empty list'}.")
        elif hasattr(calc_result_obj, 'get_impact_results'):
            try:
                print("              🔍 Attempting to use 'get_impact_results()'.")
                fetched_impact_data = calc_result_obj.get_impact_results()
                data_source_description = "'get_impact_results()'"
                if fetched_impact_data is not None:
                    print(f"              📊 '{data_source_description}' call successful, returned {'data' if len(fetched_impact_data) > 0 else 'an empty list'}.")
                else:
                    print(f"              📊 '{data_source_description}' returned None.")
            except Exception as e:
                print(f"              ⚠️ Error calling 'get_impact_results()': {e}")
                fetched_impact_data = None
                status = f"Error calling {data_source_description}"
        else:
            print("              ❌ No alternative methods ('impact_results', 'get_impact_results') yielded data.")
            if status == "Success": status = "No alternative result methods"

    if fetched_impact_data:
        # print(f"              🔬 Found {len(fetched_impact_data)} items using {data_source_description}.") # Less verbose
        for item_idx, impact_result_item in enumerate(fetched_impact_data):
            try:
                if impact_result_item is None: continue
                if not hasattr(impact_result_item, 'impact_category'): continue
                if not hasattr(impact_result_item, 'amount'): continue # Corrected to 'amount'
                impact_cat_ref = impact_result_item.impact_category
                item_numerical_value = impact_result_item.amount # Corrected to 'amount'
                if not impact_cat_ref or not hasattr(impact_cat_ref, 'id') or not impact_cat_ref.id: continue
                impact_cat_obj_from_res = client.get(olca_schema.ImpactCategory, impact_cat_ref.id)
                if not impact_cat_obj_from_res or not hasattr(impact_cat_obj_from_res, 'name'): continue
                actual_impact_cat_name = impact_cat_obj_from_res.name
                if actual_impact_cat_name == target_category_name_str:
                    carbon_footprint = item_numerical_value
                    ref_unit_name = getattr(impact_cat_obj_from_res, 'reference_unit_name', 'N/A')
                    print(f"              🎯 FOUND TARGET: {actual_impact_cat_name} - {carbon_footprint} {ref_unit_name} (Item {item_idx})")
                    status = "Success"
                    break
            except Exception: continue # Skip item on any processing error
        if carbon_footprint is None and status == "Success":
            status = "Target Category Not Found in Results"
            # print(f"              ℹ️ Target impact category '{target_category_name_str}' not found among items.") # Less verbose
    elif status == "Success":
        status = "No Impact Data List"
        # print(f"              ❌ No impact data list found from {data_source_description}.") # Less verbose
    return carbon_footprint, status

def load_process_names_from_file(filepath):
    process_names = []
    if not os.path.exists(filepath):
        print(f"❌ ERROR: Process names file not found at '{filepath}'")
        return process_names
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                name = line.strip()
                if name:
                    process_names.append(name)
        print(f"📄 Read {len(process_names)} process names from '{filepath}'.")
    except Exception as e:
        print(f"❌ ERROR: Could not read file '{filepath}': {e}")
    return process_names

def analyze_and_create_valid_systems():
    results_data = []
    client = None
    desired_process_names_count = 0
    matched_process_names_count = 0

    try:
        desired_process_names = load_process_names_from_file(PROCESS_NAMES_FILEPATH)
        desired_process_names_count = len(desired_process_names)
        if not desired_process_names:
            print("❌ No process names loaded. Cannot proceed.")
            results_data.append({"Process Name": "File Error", "Product Flow": "N/A", "Unit": "N/A", "Carbon Footprint (kg CO2 eq)": None, "Status": "No process names in file"})
            return pd.DataFrame(results_data), desired_process_names_count, matched_process_names_count

        client = olca.Client(OLCA_PORT)
        print("✅ Connected to OpenLCA")
        print("ℹ️ Fetching all process descriptors...")
        all_process_descriptors = client.get_descriptors(olca_schema.Process)
        if not all_process_descriptors:
            print("❌ ERROR: Could not fetch process descriptors.")
            results_data.append({"Process Name": "OpenLCA Error", "Product Flow": "N/A", "Unit": "N/A", "Carbon Footprint (kg CO2 eq)": None, "Status": "Failed to fetch process descriptors"})
            return pd.DataFrame(results_data), desired_process_names_count, matched_process_names_count

        process_name_to_id_map = {desc.name: desc.id for desc in all_process_descriptors}
        dynamic_target_processes = []
        missing_names_in_db_count = 0
        for name_from_file in desired_process_names:
            if name_from_file in process_name_to_id_map:
                dynamic_target_processes.append({"name": name_from_file, "id": process_name_to_id_map[name_from_file]})
            else:
                missing_names_in_db_count += 1
                # print(f"⚠️ WARNING: Name '{name_from_file}' not found in DB.") # Can be very verbose for many missing names
                results_data.append({"Process Name": name_from_file, "Product Flow": "N/A", "Unit": "N/A", "Carbon Footprint (kg CO2 eq)": None, "Status": "Name Not Found in DB"})

        matched_process_names_count = len(dynamic_target_processes)
        print(f"✅ Matched {matched_process_names_count}/{desired_process_names_count} names from file.")
        if missing_names_in_db_count > 0: print(f"ℹ️ {missing_names_in_db_count} names from file were not found in DB (first few may have been listed above).")


        if not dynamic_target_processes:
            print("❌ ERROR: No matched processes to analyze.")
            if not results_data : results_data.append({"Process Name": "Matching Error", "Product Flow": "N/A", "Unit": "N/A", "Carbon Footprint (kg CO2 eq)": None, "Status": "No names matched"})
            return pd.DataFrame(results_data), desired_process_names_count, matched_process_names_count

        lcia_method_full_obj = next((m for m in client.get_all(olca_schema.ImpactMethod) if m.name == LCIA_METHOD_NAME), None)
        if not lcia_method_full_obj:
            print(f"❌ LCIA method '{LCIA_METHOD_NAME}' not found")
            if not results_data : results_data.append({"Process Name": "LCIA Method Error", "Product Flow": "N/A", "Unit": "N/A", "Carbon Footprint (kg CO2 eq)": None, "Status": "LCIA Method Not Found"})
            return pd.DataFrame(results_data), desired_process_names_count, matched_process_names_count
        print(f"✅ Found LCIA method: {lcia_method_full_obj.name} (ID: {lcia_method_full_obj.id})")
        impact_method_ref = olca_schema.Ref(id=lcia_method_full_obj.id, name=lcia_method_full_obj.name, category=getattr(lcia_method_full_obj, 'category', None), ref_type=getattr(olca_schema.ModelType, 'IMPACT_METHOD', getattr(olca_schema.RefType, 'ImpactMethod', None)))

        for i, target_info in enumerate(dynamic_target_processes):
            target_process_id = target_info['id']
            target_process_name_from_list = target_info['name']
            print(f"\n{'='*60}\nProcessing {i+1}/{len(dynamic_target_processes)}: '{target_process_name_from_list}' (ID: {target_process_id})\n{'='*60}")
            current_status = "Starting"
            actual_process_name_from_db = "N/A"
            product_flow_name = "N/A"
            unit_name_for_df = "N/A"
            carbon_footprint_val = None
            try:
                process = client.get(olca_schema.Process, target_process_id)
                if not process: current_status = "Process Not Found by ID (Post-match)"
                else:
                    actual_process_name_from_db = process.name
                    print(f"    ✅ Retrieved DB process: {actual_process_name_from_db}")
                    if actual_process_name_from_db != target_process_name_from_list: print(f"    ⚠️ Name mismatch: File='{target_process_name_from_list}', DB='{actual_process_name_from_db}'")
                    current_status = "Process Retrieved"
                    qr_exchange = next((ex for ex in process.exchanges if getattr(ex, 'is_quantitative_reference', False) and not getattr(ex, 'is_input', True)), None)
                    if not qr_exchange:
                        qr_exchange = next((ex for ex in process.exchanges if hasattr(ex, 'is_input') and not ex.is_input), None)
                        if qr_exchange: print(f"            ⚠️ Using first output as QR (Fallback)")
                    if not qr_exchange: current_status = "No Reference Output Exchange"
                    else:
                        print(f"            ✅ Found QR: Flow='{qr_exchange.flow.name if qr_exchange.flow else 'N/A'}'")
                        current_status = "QR Found"
                        if not qr_exchange.flow or not qr_exchange.flow.id: current_status = "QR Flow Invalid"
                        else:
                            flow_of_qr = client.get(olca_schema.Flow, qr_exchange.flow.id)
                            product_flow_name = flow_of_qr.name if flow_of_qr else "Flow Error"
                            if not flow_of_qr: current_status = "Flow Error for QR"
                            else:
                                current_status = "Flow Retrieved"
                                ref_flow_property_factor = next((fpf for fpf in flow_of_qr.flow_properties if getattr(fpf, 'is_ref_flow_property', False)), None) if hasattr(flow_of_qr, 'flow_properties') and flow_of_qr.flow_properties else None
                                if not ref_flow_property_factor and hasattr(flow_of_qr, 'flow_properties') and flow_of_qr.flow_properties:
                                    ref_flow_property_factor = flow_of_qr.flow_properties[0]; print(f"                    ⚠️ Using first FlowPropertyFactor as ref.")
                                if not ref_flow_property_factor: current_status = "No Ref FlowPropertyFactor"
                                elif not ref_flow_property_factor.flow_property or not ref_flow_property_factor.flow_property.id: current_status = "Ref FlowPropertyFactor Invalid"
                                else:
                                    flow_property_ref_for_ps = ref_flow_property_factor.flow_property
                                    print(f"            ✅ FlowProperty: '{flow_property_ref_for_ps.name}'")
                                    current_status = "FlowProp Found"
                                unit_ref_for_ps = getattr(qr_exchange, 'unit', None)
                                if unit_ref_for_ps and unit_ref_for_ps.id:
                                    unit_name_for_df = unit_ref_for_ps.name if hasattr(unit_ref_for_ps, 'name') else "Unit Name?"
                                    print(f"            ✅ Unit: {unit_name_for_df}")
                                    if current_status == "FlowProp Found": current_status = "Unit&FlowProp Ready"
                                else:
                                    if current_status == "FlowProp Found": current_status = "No Unit on QR"
                        print(f"            📦 Product Flow: {product_flow_name}, 📏 Unit: {unit_name_for_df}")
                        if current_status == "Unit&FlowProp Ready":
                            print(f"            🔧 Creating product system...")
                            ps = olca_schema.ProductSystem()
                            ps_name_prefix = actual_process_name_from_db[:20].replace(' ', '_').replace('/', '_').replace('*','_')
                            ps.name = f"PS_{ps_name_prefix}_{str(uuid.uuid4())[:4]}"
                            process_ref_type = getattr(olca_schema.ModelType, 'PROCESS', getattr(olca_schema.RefType, 'Process', None))
                            ps.processes = [olca_schema.Ref(id=process.id,name=process.name,category=getattr(process, 'category', None),ref_type=process_ref_type)]
                            ps.ref_process = olca_schema.Ref(id=process.id,name=process.name,category=getattr(process, 'category', None),ref_type=process_ref_type)
                            ps.ref_exchange = olca_schema.ExchangeRef(internal_id=qr_exchange.internal_id)
                            ps.target_amount = 1.0
                            ps.target_unit = unit_ref_for_ps
                            ps.target_flow_property = flow_property_ref_for_ps
                            persisted_ps_ref = client.put(ps)
                            if not persisted_ps_ref or not persisted_ps_ref.id:
                                all_ps_desc = client.get_descriptors(olca_schema.ProductSystem)
                                persisted_ps_ref = next((d for d in all_ps_desc if d.name == ps.name), None) if all_ps_desc else None
                                if not persisted_ps_ref : raise Exception(f"Failed to create/retrieve PS '{ps.name}'")
                            print(f"            ✅ PS created: {persisted_ps_ref.name}")
                            print(f"            🧮 Setting up calculation...")

                            # --- THIS IS THE CORRECTED PART ---
                            setup = olca_schema.CalculationSetup() # Create empty instance
                            setup.calculation_type = "SIMPLE_CALCULATION"
                            setup.target = persisted_ps_ref
                            setup.impact_method = impact_method_ref
                            setup.amount = 1.0
                            setup.unit = unit_ref_for_ps
                            setup.flow_property = flow_property_ref_for_ps
                            setup.with_impacts = True
                            # --- END OF CORRECTION ---

                            print(f"            ▶️ Running calculation...")
                            calc_result = client.calculate(setup)
                            print(f"            ✅ Calc attempt finished.")
                            carbon_footprint_val, status_from_result = extract_result_safely(client, calc_result, TARGET_IMPACT_CATEGORY)
                            current_status = status_from_result
                            if carbon_footprint_val is not None and current_status == "Success":
                                print(f"            🎉 CF for {actual_process_name_from_db}: {carbon_footprint_val}")
                        elif not current_status.startswith("No ") and not current_status.startswith("QR ") and not current_status.startswith("Flow ") and not current_status.startswith("Ref "):
                             current_status = "PS Pre-req Missing"
            except Exception as process_error:
                print(f"    ❌ Error in main loop for '{target_process_name_from_list}': {process_error}")
                # traceback.print_exc() # Uncomment for very detailed error tracing if needed
                current_status = f"Process Error: {str(process_error)[:100]}" # Keep error message concise
            results_data.append({"Process Name": target_process_name_from_list, "Product Flow": product_flow_name, "Unit": unit_name_for_df, "Carbon Footprint (kg CO2 eq)": carbon_footprint_val, "Status": current_status})
    except ConnectionRefusedError:
        print(f"❌ Connection refused.")
        if not results_data: results_data.append({"Process Name": "Connection Error", "Product Flow": "N/A", "Unit": "N/A", "Carbon Footprint (kg CO2 eq)": None, "Status": "Connection Error"})
    except Exception as main_error:
        print(f"❌ Main script error: {main_error}")
        traceback.print_exc()
        if not results_data: results_data.append({"Process Name": "Script Error", "Product Flow": "N/A", "Unit": "N/A", "Carbon Footprint (kg CO2 eq)": None, "Status": f"Main Error: {str(main_error)[:100]}"})
    finally:
        if client: print("\nScript finished processing.")
    return pd.DataFrame(results_data), desired_process_names_count, matched_process_names_count

if __name__ == "__main__":
    print("🚀 OpenLCA Data Extraction - Re-Corrected CalculationSetup")
    print("=" * 60)
    results_df, total_from_file, total_matched = analyze_and_create_valid_systems()
    if not results_df.empty:
        print(f"\n📋 FINAL RESULTS")
        print("=" * 60)
        with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
            print(results_df)
        filename = "openlca_results_recorrected_calcsetup.csv" # New filename for this run
        try:
            results_df.to_csv(filename, index=False); print(f"\n💾 Saved to: {filename}")
        except Exception as e_csv: print(f"\n⚠️ Error saving CSV: {e_csv}")
        print(f"\n📊 Summary:")
        print(f"  - Names read from file: {total_from_file}")
        print(f"  - Names matched to DB: {total_matched}")
        successful_calculations = len(results_df[results_df['Status'] == 'Success'])
        print(f"  - Successful calculations: {successful_calculations} / {total_matched if total_matched > 0 else 'N/A attempted'}")
        status_counts = results_df['Status'].value_counts()
        if not status_counts.empty:
            print("  - Breakdown of all statuses:")
            for status_name, count_val in status_counts.items(): print(f"    - {status_name}: {count_val}")
    else:
        print("❌ No results (or initial file/DB error).")


🚀 OpenLCA Data Extraction - Re-Corrected CalculationSetup
📄 Read 957 process names from 'all_process_names.txt'.
✅ Connected to OpenLCA
ℹ️ Fetching all process descriptors...
✅ Matched 891/957 names from file.
ℹ️ 66 names from file were not found in DB (first few may have been listed above).
✅ Found LCIA method: IPCC 2021 (ISO 14067) (ID: 437c5849-901b-4eba-af18-211f849bde6a)

Processing 1/891: '6000-Q GasCogenerate Power Used in Place/MJ S' (ID: ea9c9c23-7db9-3a2a-b3f1-7a68d3fc7c6e)
    ✅ Retrieved DB process: 6000-Q GasCogenerate Power Used in Place/MJ S
            ✅ Found QR: Flow='6000-Q GasCogenerate Power Used in Place/MJ S'
            ✅ FlowProperty: 'Energy'
            ✅ Unit: MJ
            📦 Product Flow: 6000-Q GasCogenerate Power Used in Place/MJ S, 📏 Unit: MJ
            🔧 Creating product system...
            ✅ PS created: PS_6000-Q_GasCogenerate_e5ac
            🧮 Setting up calculation...
            ▶️ Running calculation...
            ✅ Calc attempt finished.
   

ERROR:root:request total-impacts failed: 500: Failed to call method public org.openlca.ipc.RpcResponse org.openlca.ipc.handlers.ResultHandler.getTotalImpacts(org.openlca.ipc.RpcRequest): null


              🔍 Attempting to use 'get_total_impacts()'.
              📊 ''get_total_impacts()'' call successful, returned an empty list.
              ℹ️  Trying alternative methods as primary failed/yielded no data.
              ❌ No alternative methods ('impact_results', 'get_impact_results') yielded data.

Processing 3/891: '6142-Q BioFuel Power Used in Place/MJ S' (ID: d8c9eedf-a3fc-31a6-b1b8-29ab1d76ea18)
    ✅ Retrieved DB process: 6142-Q BioFuel Power Used in Place/MJ S
            ✅ Found QR: Flow='6142-Q BioFuel Power Used in Place/MJ S'
            ✅ FlowProperty: 'Energy'
            ✅ Unit: MJ
            📦 Product Flow: 6142-Q BioFuel Power Used in Place/MJ S, 📏 Unit: MJ
            🔧 Creating product system...
            ✅ PS created: PS_6142-Q_BioFuel_Power_dfea
            🧮 Setting up calculation...
            ▶️ Running calculation...
            ✅ Calc attempt finished.
              ⏳ Waiting for calculation result to be ready...
              🔍 Attempting to us

ERROR:root:request total-impacts failed: 500: Failed to call method public org.openlca.ipc.RpcResponse org.openlca.ipc.handlers.ResultHandler.getTotalImpacts(org.openlca.ipc.RpcRequest): null


              🔍 Attempting to use 'get_total_impacts()'.
              📊 ''get_total_impacts()'' call successful, returned an empty list.
              ℹ️  Trying alternative methods as primary failed/yielded no data.
              ❌ No alternative methods ('impact_results', 'get_impact_results') yielded data.

Processing 7/891: '6266-Q Wind Power Used  in Place*MJ S' (ID: 70e706c6-e9ad-34c7-a63e-aa6b792a5bff)
    ✅ Retrieved DB process: 6266-Q Wind Power Used  in Place*MJ S
            ✅ Found QR: Flow='6266-Q Wind Power Used  in Place*MJ S'
            ✅ FlowProperty: 'Energy'
            ✅ Unit: MJ
            📦 Product Flow: 6266-Q Wind Power Used  in Place*MJ S, 📏 Unit: MJ
            🔧 Creating product system...
            ✅ PS created: PS_6266-Q_Wind_Power_Us_1185
            🧮 Setting up calculation...
            ▶️ Running calculation...
            ✅ Calc attempt finished.
              ⏳ Waiting for calculation result to be ready...


ERROR:root:request total-impacts failed: 500: Failed to call method public org.openlca.ipc.RpcResponse org.openlca.ipc.handlers.ResultHandler.getTotalImpacts(org.openlca.ipc.RpcRequest): null


              🔍 Attempting to use 'get_total_impacts()'.
              📊 ''get_total_impacts()'' call successful, returned an empty list.
              ℹ️  Trying alternative methods as primary failed/yielded no data.
              ❌ No alternative methods ('impact_results', 'get_impact_results') yielded data.

Processing 8/891: '6331-AU Generate Steam on Site/MJ S' (ID: 9d8a4c10-cbd5-3416-84df-f169f0890dca)
    ✅ Retrieved DB process: 6331-AU Generate Steam on Site/MJ S
            ✅ Found QR: Flow='6331-AU Generate Steam on Site/MJ S'
            ✅ FlowProperty: 'Energy'
            ✅ Unit: MJ
            📦 Product Flow: 6331-AU Generate Steam on Site/MJ S, 📏 Unit: MJ
            🔧 Creating product system...
            ✅ PS created: PS_6331-AU_Generate_Ste_efc7
            🧮 Setting up calculation...
            ▶️ Running calculation...
            ✅ Calc attempt finished.
              ⏳ Waiting for calculation result to be ready...
              🔍 Attempting to use 'get_total_imp